# Fase 0 · Lunes 16 nov 2026 · Bloque de 2 h · Arrays & Hashing

**Regla nueva: concepto antes, solución nunca.** Primero aprendes el patrón. Después lo aplicas **sin pistas**.

| Bloque | Tiempo | Qué toca |
|---|---|---|
| A · Aprender | 25 min | Los 4 patrones de Arrays & Hashing + calentamiento con la API de Python |
| B · Corregir | 15 min | Lo pendiente del sábado: el tipo de retorno, el `print` y las explicaciones en inglés |
| C · Aplicar | 60 min | Contains Duplicate y Valid Anagram: cronometrados, **sin pistas**, tope de 20 min cada uno |
| D · Reto | 15 min | Top K en O(n) con bucket sort (opcional; no cuenta para la línea base) |
| Cierre | 5 min | Registrar las horas en `private/log.md` |

In [1]:
import time
from typing import List

def start():
    """Llama start() al empezar un problema; guarda el valor que devuelve."""
    return time.time()

def stop(t0):
    """Llama stop(t0) al terminar; imprime los minutos que tardaste."""
    mins = (time.time() - t0) / 60
    print(f"⏱  {mins:.1f} min")
    return round(mins, 1)

---
# Bloque A · Aprender el patrón (25 min)

La idea central de casi todo Arrays & Hashing es **cambiar espacio por tiempo**. En lugar de volver a recorrer la lista para buscar algo, que es O(n) por cada búsqueda y O(n²) en total, guardas lo que ya viste en una estructura donde buscar cuesta **O(1)**: un **hash set** o un **hash map**.

## A.1 · Hash set: "¿ya vi esto?"

Un `set` guarda elementos **sin repetir**. Preguntar `x in mi_set` cuesta O(1) en promedio, mientras que `x in mi_lista` cuesta O(n).

| Operación | Sintaxis | Costo |
|---|---|---|
| Crear vacío | `s = set()` | O(1) |
| Agregar | `s.add(x)` | O(1) |
| ¿Está? | `x in s` | O(1) |
| Desde una lista | `set(lista)` | O(n) |

**Cuándo pensar en un set:** cuando el problema dice "duplicado", "único", "ya apareció" o "visitado".

**En inglés:** *"I use a hash set to track what I've already seen, so each membership check is O(1)."*

### Calentamiento A.1 (5 min)
Recibes la lista de ciudades que visitó un viajero, **en orden**. Devuelve la **primera ciudad que visitó por segunda vez**, o `None` si nunca repitió ninguna.

Usa un `set`. No uses `.count()` ni recorras la lista dos veces.

In [3]:
def first_revisited(cities: List[str]):
    visit = set()
    for c in cities:
        if c in visit:
            return c
        else:
            visit.add(c)
    
print (first_revisited(["Bogotá", "Cali", "Medellín", "Cali", "Bogotá"]))
assert first_revisited(["Bogotá", "Cali", "Medellín", "Cali", "Bogotá"]) == "Cali"
assert first_revisited(["Bogotá", "Cali"]) is None
assert first_revisited([]) is None
print("✅ A.1")

Cali
✅ A.1


## A.2 · Hash map (dict): key → value

Un `dict` asocia una key con un value. Buscar, insertar y actualizar por key cuesta O(1).

Tiene dos usos principales:

**1. Lookup ("¿ya vi lo que me falta?").** Guardas lo que viste junto con información útil, como su índice, para encontrarlo después. Así resolviste Two Sum.

**2. Frequency count ("¿cuántas veces aparece cada cosa?").** La key es el elemento y el value es cuántas veces aparece. Hay tres formas de escribirlo, de la más manual a la más idiomática:

| Forma | Idea |
|---|---|
| `if x in d: ... else: ...` | La que usaste el sábado: funciona, pero es larga |
| `d[x] = d.get(x, 0) + 1` | `get` devuelve un valor por defecto si la key no existe |
| `collections.Counter(iterable)` | Una clase hecha para contar; también se puede comparar con `==` |

**Ojo con la complejidad de `Counter`:** construirlo cuesta O(n), igual que contar a mano. No es "gratis", solo es más corto. En entrevista puedes usarlo, pero tienes que saber qué hace por dentro.

**En inglés:** *"I build a frequency map in one pass, O(n) time, where each key is an element and the value is its count."*

### Calentamiento A.2 (7 min)
Cuenta cuántas veces aparece cada palabra en una frase (en minúsculas y separada por espacios).

1. Escribe `word_count_get` usando `dict.get`.
2. Escribe `word_count_counter` usando `Counter`.
3. Comprueba que las dos dan el mismo resultado.

In [21]:
from collections import Counter

def word_count_get(sentence: str) -> dict:
    repit = {}
    sentence = sentence.split()
    for s in sentence:
        repit[s] = repit.get(s,0) + 1
    return repit

def word_count_counter(sentence: str) -> dict:
    sentence = sentence.split()
    c = Counter(sentence)
    return c
    
frase = "el gol del año fue el gol de james"
word_count_counter(frase)
assert word_count_get(frase)["gol"] == 2
assert word_count_get(frase)["el"] == 2
assert word_count_get(frase)["james"] == 1
assert word_count_get(frase) == word_count_counter(frase)
assert word_count_get("") == {} or word_count_get("") == {"": 1}  # piensa: ¿qué devuelve "".split()?
print("✅ A.2")

✅ A.2


## A.3 · Canonical key: agrupar cosas "equivalentes"

Cuando tienes que **agrupar** elementos que son "iguales" según alguna regla, la idea es transformar cada uno en una **key canónica**: una representación que sale idéntica para todos los que son equivalentes. Después agrupas con un `dict` de `key → lista`.

- En Group Anagrams, la key era el conteo de letras como `tuple`.
- Otra opción es el string ordenado: `"".join(sorted(s))`. Cuesta O(n log n) por palabra en lugar de O(n).

**¿Por qué `tuple` y no `list`?** Las keys de un `dict` tienen que ser **hashable**, es decir, inmutables. Una `list` se puede modificar, así que no sirve como key. Una `tuple` sí.

**`defaultdict(list)`** crea la lista vacía automáticamente la primera vez que usas una key, así que no necesitas el `if key not in d`.

**Cuidado con lo que devuelves:** `d.values()` es una *view* de tipo `dict_values`, no una `list`. Si la firma promete `List[...]`, tienes que convertirla.

**En inglés:** *"I map each item to a canonical key, so equivalent items collide on the same key, and group them with a defaultdict of lists."*

### Calentamiento A.3 (5 min)
Agrupa equipos de fútbol por la **cantidad de letras** de su nombre. Usa `defaultdict(list)`. Devuelve una `list` de listas, no `dict_values`.

In [20]:
from collections import defaultdict

def group_by_length(names: List[str]) -> List[List[str]]:
    groups = defaultdict(list)
    for n in names:
        groups[len(n)].append(n)
    return list(groups.values())

r = group_by_length(["Nacional", "Millos", "América", "Junior", "Santa Fe"])
assert isinstance(r, list), "Devuelve una list, no dict_values"
assert sorted(sorted(g) for g in r) == sorted(sorted(g) for g in [["Millos", "Junior"], ["América"], ["Nacional", "Santa Fe"]])
print("✅ A.3")

✅ A.3


## A.4 · Bucket sort: cuando el rango de valores está acotado

Ordenar cuesta O(n log n). Pero si sabes que los valores **solo pueden estar en un rango pequeño y conocido**, por ejemplo de 0 a n, puedes saltarte el ordenamiento:

1. Creas una lista de "cubetas" (*buckets*), una por cada valor posible.
2. Pones cada elemento en la cubeta que corresponde a su valor: **el valor se usa como índice**.
3. Recorres las cubetas en orden, o al revés si quieres los mayores primero.

Eso es O(n + rango). Si el rango es O(n), el total es **O(n)**.

**Cómo crear la lista de cubetas:** usa `[[] for _ in range(tamaño)]`. **No** uses `[[]] * tamaño`: eso crea N referencias a *la misma* lista. Compruébalo tú mismo en la celda de abajo.

**En inglés:** *"Since the values are bounded by n, I can use them as indices into an array of buckets instead of sorting, which brings it down to O(n)."*

In [22]:
# Experimento: corre esto y explica qué pasó
malo = [[]] * 3
malo[0].append("x")
print("malo :", malo)

bueno = [[] for _ in range(3)]
bueno[0].append("x")
print("bueno:", bueno)

malo : [['x'], ['x'], ['x']]
bueno: [['x'], [], []]


### Calentamiento A.4 (5 min)
Las notas de un parcial son enteros de 0 a 5. Recibes una lista de tuplas `(estudiante, nota)`. Devuelve los nombres **ordenados de la nota más alta a la más baja**, usando buckets. **Sin** `sorted` ni `.sort()`.

Dentro de una misma nota, conserva el orden de llegada.

In [24]:
def rank_students(grades: list) -> List[str]:
    bucket = [[] for _ in range(6)]
    for name, grade in grades:
        bucket[grade].append(name)
    sort = []
    for b in reversed(bucket):
        sort.extend(b)
    return sort
g = [("Ana", 3), ("Luis", 5), ("Sara", 0), ("Juan", 5), ("Leo", 3)]
assert rank_students(g) == ["Luis", "Juan", "Ana", "Leo", "Sara"]
assert rank_students([]) == []
print("✅ A.4")

✅ A.4


## A.5 · Protocolo para no quedarte en blanco (los primeros 10 min de cada problema)

Antes de escribir código, llena esto **en papel o en la celda de "Protocolo"**:

1. **Con mis palabras:** ¿qué entra?, ¿qué sale?
2. **Ejemplos a mano:** uno normal y uno de borde (vacío, un solo elemento, repetidos). Fíjate en qué haces tú con la cabeza: ese suele ser el algoritmo.
3. **Brute force:** escribe la solución lenta y su complejidad. Una solución lenta vale más que una página en blanco.
4. **Optimizar:** ¿qué operación repito? ¿Cuál de los patrones A.1 a A.4 la hace O(1)?

**Frase de entrevista:** *"Let me start with a brute force approach, then optimize."*

---
# Bloque B · Corregir lo del sábado (15 min)

Haz los cambios **en el notebook del sábado** (`2026-11-14_fase0_sabado.ipynb`) y vuelve a correr sus pruebas:

- [ ] `group_anagrams`: que devuelva una `list`, no `dict_values` (mira A.3).
- [ ] `group_anagrams`: complejidad completa en el comentario. ¿Qué son *m* y *n*? ¿Cuánto espacio usa?
- [ ] `top_k_frequent`: quita el `print(freq)` y cambia el conteo por `dict.get` o `Counter` (mira A.2).
- [ ] `top_k_frequent`: complejidad de espacio en el comentario.
- [ ] Las **explicaciones en inglés** de Group Anagrams y Top K. Dilas primero en voz alta y después escríbelas.

Pega aquí tus dos explicaciones en inglés para que te corrija:

**Group Anagrams (EN):**

**Top K Frequent (EN):**

---
# Bloque C · Aplicar sin pistas (60 min)

**Reglas:** cronómetro, tope de 20 min por problema y **cero pistas**. Si no sale en 20 min, lo marcas como "parcial" o "no", anotas dónde te trabaste y pasas al siguiente. Un "no" también es un dato.

## C.1 · Contains Duplicate (Easy)

Recibes una lista de enteros `nums`. Devuelve `True` si **algún valor aparece al menos dos veces**, y `False` si todos son distintos.

**Ejemplo:** `[1, 2, 3, 3]` → `True` · `[1, 2, 3, 4]` → `False`

**Meta:** tiempo O(n).

In [25]:
t0 = start()

**Protocolo (10 min):**
1. Con mis palabras: 
2. Ejemplos a mano (normal + borde):
3. Brute force y su complejidad:
4. ¿Qué repito? ¿Qué patrón lo hace O(1)?

In [28]:
def contains_duplicate(nums: List[int]) -> bool:
    # Approach:
    s = set()
    for n in nums:
        if n in s:
            return True
        else: 
            s.add(n)
    return False
    # Time complexity:
    # Space complexity:

In [29]:
# --- Pruebas: no las modifiques ---
def test_contains_duplicate():
    assert contains_duplicate([1, 2, 3, 3]) is True
    assert contains_duplicate([1, 2, 3, 4]) is False
    assert contains_duplicate([]) is False
    assert contains_duplicate([7]) is False
    assert contains_duplicate([-1, -1]) is True
    assert contains_duplicate([0, 1, 0]) is True
    big = list(range(200_000))
    assert contains_duplicate(big) is False
    assert contains_duplicate(big + [199_999]) is True
    print("✅ contains_duplicate: todas las pruebas pasan")

test_contains_duplicate()

✅ contains_duplicate: todas las pruebas pasan


In [30]:
tiempo_contains_duplicate = stop(t0)

⏱  2.7 min


### Reflexión · Contains Duplicate
- **¿Lo resolví?** (sí / parcial / no): Si 
- **Tiempo:** 2.7 min
- **Enfoque y complejidad (tiempo y espacio):** O(1)
- **Pregunta extra:** ¿hay una solución en una sola línea? ¿Tiene la misma complejidad? ¿Cuál es mejor si el duplicado aparece al principio de una lista de 1 millón de elementos? 
- **Dónde me trabé:** No me trabe
- **Explicación en inglés:** I use a hashed set to have the numbers I was obtaining and if the number I was currently in was already in the set then I returned True if after I went through all the list there was not a match I return False.

## C.2 · Valid Anagram (Easy)

Recibes dos strings, `s` y `t`. Devuelve `True` si `t` es un **anagrama** de `s`, es decir, si tiene exactamente las mismas letras con la misma frecuencia.

**Ejemplo:** `s = "racecar"`, `t = "carrace"` → `True` · `s = "jar"`, `t = "jam"` → `False`

**Meta:** tiempo O(n + m).

In [31]:
t0 = start()

**Protocolo (10 min):**
1. Con mis palabras:
2. Ejemplos a mano (normal + borde):
3. Brute force y su complejidad:
4. ¿Qué repito? ¿Qué patrón lo hace O(1)?

In [ ]:
from collections import Counter
def is_anagram(s: str, t: str) -> bool:
    # Approach:
    s_counter = Counter(s)
    t_counter = Counter(t)

    if s_counter == t_counter:
        return True
    else:
        return False
    
    # Time complexity: 
    # Space complexity:

is_anagram("racecar", "carrace")

True

In [34]:
# --- Pruebas: no las modifiques ---
def test_is_anagram():
    assert is_anagram("racecar", "carrace") is True
    assert is_anagram("jar", "jam") is False
    assert is_anagram("", "") is True
    assert is_anagram("a", "ab") is False
    assert is_anagram("aab", "abb") is False   # mismas letras, distinta frecuencia
    assert is_anagram("listen", "silent") is True
    assert is_anagram("ab", "a") is False
    big = "abc" * 100_000
    assert is_anagram(big, big[::-1]) is True
    print("✅ is_anagram: todas las pruebas pasan")

test_is_anagram()

✅ is_anagram: todas las pruebas pasan


In [35]:
tiempo_valid_anagram = stop(t0)

⏱  5.8 min


### Reflexión · Valid Anagram
- **¿Lo resolví?** (sí / parcial / no): Si
- **Tiempo:** 5.8
- **Enfoque y complejidad (tiempo y espacio):** 
- **Pregunta extra:** si los strings pudieran tener cualquier carácter Unicode (tildes, emojis), ¿seguiría sirviendo un arreglo de 26 posiciones? ¿Qué usarías?
- **Pregunta extra 2:** ¿qué comprobación barata puedes hacer al principio para salir rápido? usar Counter
- **Dónde me trabé:** No me trabe
- **Explicación en inglés:** I used a hashed map to count the caracters in the words and have the caracter as a key then compared the hashed maps

---
# Bloque D · Reto: Top K en O(n) (15 min, opcional)

Rehaz `top_k_frequent` **sin** `sorted`, `.sort()`, `heapq` ni `Counter.most_common`. Usa bucket sort (mira A.4).

Pregunta guía: ¿cuál es la frecuencia **máxima** posible en una lista de largo n? ¿Cuántas cubetas necesitas entonces?

In [ ]:
t0 = start()

In [55]:
def top_k_frequent_linear(nums: List[int], k: int) -> List[int]:
    # Approach:
    bucket = [[] for _ in range(len(nums) + 1)]
    count = Counter(nums)
    for n, f in count.items():
        bucket[f].append(n)
    sort = []
    for b in reversed(bucket):
        sort.extend(b)
        if k == len(sort):
            return sort
    return sort[:k]
    # Time complexity:
    # Space complexity:
top_k_frequent_linear([1, 2, 2, 3, 3, 3], 2)
top_k_frequent_linear([1, 1, 2, 2, 3], 1)   # devuelve [1, 2, 3]


[1]

In [54]:
# --- Pruebas: no las modifiques ---
def test_top_k_linear():
    assert sorted(top_k_frequent_linear([1, 2, 2, 3, 3, 3], 2)) == [2, 3]
    assert sorted(top_k_frequent_linear([7, 7], 1)) == [7]
    assert sorted(top_k_frequent_linear([1], 1)) == [1]
    assert sorted(top_k_frequent_linear([4, 4, 4, -1, -1, 2], 2)) == [-1, 4]
    assert sorted(top_k_frequent_linear([5, 6, 7], 3)) == [5, 6, 7]
    big = [1] * 50_000 + [2] * 30_000 + list(range(3, 10_003))
    assert sorted(top_k_frequent_linear(big, 2)) == [1, 2]
    print("✅ top_k_frequent_linear: todas las pruebas pasan")

test_top_k_linear()

✅ top_k_frequent_linear: todas las pruebas pasan


In [ ]:
tiempo_top_k_linear = stop(t0)

### Reflexión · Top K en O(n)
- **¿Lo logré?:** Si
- **Tiempo:**
- **¿Por qué es O(n) y no O(n log n)?** (explícalo en inglés):

---
# Cierre del día

In [ ]:
print("Resumen de hoy")
for nombre in ["tiempo_contains_duplicate", "tiempo_valid_anagram", "tiempo_top_k_linear"]:
    print(f"  {nombre}: {globals().get(nombre, '—')} min")

| Bloque | Planeado | Real | Completado |
|---|---|---|---|
| A · Aprender | 25 min | 40 min | si |
| B · Corregir | 15 min | 10 min | Si |
| C · Aplicar | 60 min | 20 min | Si |
| D · Reto | 15 min | 15 min | Si |
| **Total** | **~2 h** | 1h 25min | Si |

- **Línea base NeetCode:** 5 / 10 (meta de hoy: 5)
- **¿Usé alguna pista?:** Si
- **¿Anoté las horas en `private/log.md`?** (sábado, domingo y hoy): No
- **¿Commit y push?:** Listo
- **Mañana martes:** Agent loop, empezando por el diseño en papel (2.0 del notebook del sábado).